In [ ]:
# Colab/bootstrap: clone this repository and install it editable.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
MARK_REL = Path("src") / "mindscopex_analysis" / "__init__.py"


def find_repo_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / MARK_REL).is_file():
            return path
    return None


root = find_repo_root()
if root is None:
    workdir = Path(os.environ.get("COLAB_REPO_DIR", "/content/mindscopex_analysis"))
    if (workdir / MARK_REL).is_file():
        subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=False)
        root = workdir
    else:
        workdir.parent.mkdir(parents=True, exist_ok=True)
        if workdir.exists():
            shutil.rmtree(workdir)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(workdir)])
        root = workdir

os.environ["MINDSCOPEX_ROOT"] = str(root.resolve())
os.chdir(root)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
print("ready:", root)


# 09. Token Position Sweep

질문: feature를 어느 prompt token 위치에서 제거해야 효과가 나는가?

기본 실험은 마지막 prompt token `Answer:` 위치에 개입합니다. 여기서는 prompt 끝 주변 여러 token에 같은 direction을 넣어봅니다.

In [ ]:
from pathlib import Path
import os
import sys

root = Path(os.environ.get("MINDSCOPEX_ROOT", Path.cwd())).resolve()
if not (root / "src" / "mindscopex_analysis" / "__init__.py").is_file():
    for candidate in [root, *root.parents]:
        if (candidate / "src" / "mindscopex_analysis" / "__init__.py").is_file():
            root = candidate
            break
    else:
        raise RuntimeError("Could not find repository root. Run the clone cell first.")

src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(root)


In [ ]:
import torch
from IPython.display import display

from mindscopex_analysis import (
    BAT_BALL_CASE,
    DEFAULT_MODEL_ID,
    DEFAULT_QWEN_SCOPE_REPO_ID,
    LureCase,
    answer_logprob_margin,
    answer_variant_rows,
    bat_ball_answer_variants,
    bat_ball_paraphrases,
    candidate_feature_rows,
    case_transfer_rows,
    coefficient_sweep_for_handle,
    control_delta_bypass_rows,
    crt_transfer_cases,
    decoder_cosine_rows,
    default_sae_device,
    dtype_from_name,
    feature_handle_from_result,
    intervention_mode_rows,
    layer_feature_search_rows,
    load_or_discover_handle_and_sae,
    load_qwen_language_model,
    load_qwen_scope_sae,
    prompt_token_window_rows,
    rank_lure_feature_effects,
    recommended_dtype_name,
    sae_decoder_direction,
    save_feature_handle,
    semantic_lure_cases,
    token_position_sweep_rows,
)

MODEL_ID = DEFAULT_MODEL_ID
SAE_REPO_ID = DEFAULT_QWEN_SCOPE_REPO_ID
DTYPE = recommended_dtype_name()
SAE_DEVICE = default_sae_device()
SAE_DTYPE = DTYPE
HANDLE_CACHE = root / "outputs" / "candidates" / "bat_ball_top_feature.json"

lm = load_qwen_language_model(MODEL_ID, device_map="auto", dtype=DTYPE, dispatch=True)
print({"model": MODEL_ID, "sae_repo": SAE_REPO_ID, "dtype": DTYPE, "sae_device": SAE_DEVICE})


In [ ]:
CASE = BAT_BALL_CASE
REFRESH_FEATURE = False

handle, sae, discovery_rows, loaded_from_cache = load_or_discover_handle_and_sae(
    lm,
    CASE,
    repo_id=SAE_REPO_ID,
    cache_path=HANDLE_CACHE,
    default_layer=14,
    sae_device=SAE_DEVICE,
    sae_dtype=dtype_from_name(SAE_DTYPE),
    top_n=12,
    refresh=REFRESH_FEATURE,
)

print("feature loaded from cache:", loaded_from_cache)
display(handle.as_row())
if discovery_rows:
    display(discovery_rows[:12])


In [ ]:
window = prompt_token_window_rows(lm.tokenizer, CASE.prompt, window=10)
display(window)

token_indices = [row["token_index"] for row in window]
rows = token_position_sweep_rows(
    lm,
    CASE,
    layer=handle.layer,
    sae=sae,
    feature_id=handle.feature_id,
    feature_value=handle.feature_value,
    token_indices=token_indices,
    coefficient=1.0,
    intervention_mode="remove_activation",
)
display(rows)


해석 체크: 마지막 prompt token에서만 효과가 크면 answer-selection feature일 수 있습니다. 문제 본문 token에서도 효과가 있으면 feature가 문제 해석 중간에 관여할 수 있습니다.